# 04c Build Screened Dataset

Apply the configured prediction source, System 1 run ID, and probability threshold to saved predictions. This stage is offline and always replaces the canonical screened parquet from the selected prediction store.

## Inputs

- Candidate rows from `candidate_amr_pair_year_counts.parquet`.
- The prediction store selected by `screening.active_source`.
- For System 1, `system1.run_id` and `screening.system1_probability_threshold`; for LLM, the configured included decisions.

## Outputs

- Rebuilt `yearly_amr_screened_pairs.parquet`, replaced from the selected saved predictions without model calls.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config

from amr.screening import build_yearly_screened_pairs


In [ ]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
OUTPUT_YEARLY_AMR_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
OUTPUT_YEARLY_SCREEN_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
OUTPUT_UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
OUTPUT_SYSTEM1_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.system1_decisions", PROJECT_ROOT)
ACTIVE_PREDICTION_SOURCE = str(cfg(PIPELINE_CONFIG, "screening.active_source", "llm")).lower()
SYSTEM1_PROBABILITY_THRESHOLD = float(cfg(PIPELINE_CONFIG, "screening.system1_probability_threshold", 0.5))
SYSTEM1_RUN_ID = str(cfg(PIPELINE_CONFIG, "system1.run_id", "jev-default"))
SCREENED_INCLUDE_DECISIONS = tuple(cfg(PIPELINE_CONFIG, "llm.screened_include_decisions", ["strict", "yes"]))
if ACTIVE_PREDICTION_SOURCE not in {"llm", "system1"}:
    raise ValueError("screening.active_source must be 'llm' or 'system1'.")
if not 0.0 <= SYSTEM1_PROBABILITY_THRESHOLD <= 1.0:
    raise ValueError("screening.system1_probability_threshold must be between 0 and 1.")


## Rebuild canonical screened output from disk

In [ ]:
if not OUTPUT_YEARLY_AMR_PATH.exists():
    raise FileNotFoundError(f"Missing {OUTPUT_YEARLY_AMR_PATH}. Run notebook 04 collection first.")
yearly_amr_df = pd.read_parquet(OUTPUT_YEARLY_AMR_PATH)
if ACTIVE_PREDICTION_SOURCE == "llm":
    prediction_path = OUTPUT_UNIFIED_DECISIONS_PATH
    builder_kwargs = {"include_decisions": SCREENED_INCLUDE_DECISIONS}
else:
    prediction_path = OUTPUT_SYSTEM1_DECISIONS_PATH
    builder_kwargs = {
        "probability_threshold": SYSTEM1_PROBABILITY_THRESHOLD,
        "system1_run_id": SYSTEM1_RUN_ID,
    }
if not prediction_path.exists():
    raise FileNotFoundError(f"Missing selected prediction store: {prediction_path}. Run notebook 04b first.")
prediction_df = pd.read_parquet(prediction_path)
print(f"Prediction source: {ACTIVE_PREDICTION_SOURCE}; rows loaded: {len(prediction_df):,}")
if ACTIVE_PREDICTION_SOURCE == "system1":
    print(f"System 1 run: {SYSTEM1_RUN_ID}; P(yes) threshold: {SYSTEM1_PROBABILITY_THRESHOLD:.3f}")

yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    prediction_df,
    prediction_source=ACTIVE_PREDICTION_SOURCE,
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=False,
    **builder_kwargs,
)
print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {len(yearly_screened_df[['pathogen', 'antibiotic']].drop_duplicates()):,}")
print(f"Screened output replaced at: {OUTPUT_YEARLY_SCREEN_PATH}")
yearly_screened_df.head(10)
